In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd

from frigeo import racine_projet
from frigeo.perimetre import decouvrir

# PARAMETRE A MODIFIER CHAQUE MOIS
# Dernier mois déposé dans data/raw, au format AAAA-MM
PERIODE_FIN = "2026-03"

In [ ]:
DATA = racine_projet() / "data"
perimetre = decouvrir(DATA, PERIODE_FIN)
perimetre.verifier()
print(perimetre.resume())
perimetre.tableau()

In [ ]:
import csv
import io
from collections import Counter


def format_declare(nom):
    """Renvoie (flux, encodage, séparateur) selon le dossier de données."""
    if nom.startswith("pieces_interventions"):
        return "pieces_interventions", "utf-8", ";"
    if nom.startswith("interventions"):
        return "interventions", "utf-8", ";"
    if nom.startswith(("FEC_extraction", "999000001FEC")):
        return "fec", "iso-8859-15", "\t"
    if nom == "clients.csv":
        return "clients", "cp1252", ";"
    if nom == "fournisseurs.csv":
        return "fournisseurs", "utf-8", ";"
    return None


def sonder_texte(chemin, encodage, sep):
    brut = (DATA / chemin).read_bytes()
    res = {
        "bom_utf8": brut.startswith(b"\xef\xbb\xbf"),
        "crlf": brut.count(b"\r\n"),
        "lf_seul": brut.count(b"\n") - brut.count(b"\r\n"),
        "decodage": "ok",
    }
    try:
        texte = brut.decode(encodage)
    except UnicodeDecodeError as e:
        res["decodage"] = f"echec a l'octet {e.start}"
        return res, None
    lecteur = csv.reader(io.StringIO(texte, newline=""), delimiter=sep)
    entete = next(lecteur, [])
    longueurs = Counter(len(ligne) for ligne in lecteur)
    res.update(
        colonnes_entete=len(entete),
        enregistrements=sum(longueurs.values()),
        lignes_physiques=lecteur.line_num,
        longueurs=dict(longueurs.most_common(4)),
    )
    return res, tuple(entete)


lignes, entetes = [], {}
for chemin in perimetre:
    fmt = format_declare(Path(chemin).name)
    if fmt is None:
        continue
    flux, encodage, sep = fmt
    res, entete = sonder_texte(chemin, encodage, sep)
    lignes.append({"fichier": chemin, "flux": flux, **res})
    entetes.setdefault((flux, entete), []).append(chemin)

structure_texte = pd.DataFrame(lignes)
display(structure_texte)

print("En-têtes distincts par flux")
for (flux, entete), fichiers in entetes.items():
    print(flux, len(fichiers), "fichier(s)", "|".join(entete or ()))

In [ ]:
import openpyxl

lignes, entetes = [], {}
for chemin in perimetre:
    if not chemin.endswith(".xlsx"):
        continue
    classeur = openpyxl.load_workbook(DATA / chemin, read_only=True, data_only=True)
    for feuille in classeur.worksheets:
        entete = next(feuille.iter_rows(min_row=1, max_row=1, values_only=True), ())
        lignes.append(
            {
                "fichier": chemin,
                "feuille": feuille.title,
                "lignes": feuille.max_row,
                "colonnes": feuille.max_column,
            }
        )
        entetes.setdefault((feuille.title, tuple(entete)), []).append(chemin)
    classeur.close()

structure_excel = pd.DataFrame(lignes)
display(structure_excel)

print("En-têtes distincts par feuille")
for (feuille, entete), fichiers in entetes.items():
    print(feuille, len(fichiers), "fichier(s)", "|".join(str(c) for c in entete))